In [1]:
from dotenv import load_dotenv, find_dotenv
import os
from langchain.tools import tool
from typing import Literal
from pydantic import BaseModel, Field

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
env_path = find_dotenv()
load_dotenv(env_path, override=True)

True

In [3]:
@tool
def search_database(query: str, limit: int=10) -> str:
    """
    Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    database = [
        {"id":1, "name": "John Doe", "email": "john@pytechie.com"},
        {"id":2, "name": "Jane Smith", "email": "jane@pytechie.com"},
        {"id":3, "name": "Alice Johnson", "email": "alice@pytechie.com"},
        {"id":4, "name": "Bob Brown", "email": "bob@pytechie.com"}
    ]

    results = []

    for record in database:
        if query.lower() in record.get("name").lower():
            results.append(record)

        if len(results) >= limit:
            break

    if not results:
        return "No mtaching records found."
    
    output = "\n".join(
        [
            f"ID: {r.get('id')}, Name: {r.get('name')}, Email: {r.get('email')}" for r in results
        ]
    )

    return output



In [4]:
search_database.invoke(
    {
        "query": "john",
        "limit": 3
    }
)

'ID: 1, Name: John Doe, Email: john@pytechie.com\nID: 3, Name: Alice Johnson, Email: alice@pytechie.com'

In [5]:
print(search_database.name)
print(search_database.description)

search_database
Search the customer database for records matching the query.

Args:
    query: Search terms to look for
    limit: Maximum number of results to return


In [6]:
@tool(
    "calculator",
    description="Performs arithmetic calculations. Use this for math problems."
)
def calc(expression: str) -> str:
    """
    Evaluate mathematical expressions.

    Args:
        expression: Mathematical expression to evaluate
    """

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Error: {str(e)}"


In [7]:
calc.invoke(
    {
        "expression": "10 + 20 * 3"
    }
)

'70'

In [8]:
print(calc.name)
print(calc.description)

calculator
Performs arithmetic calculations. Use this for math problems.


In [9]:
class WeatherInput(BaseModel):
    """Input schema for weather queries."""

    location: str = Field(
        description="City name or geographic coordinates"
    )

    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )

    include_forecast: bool = Field(
        default=False,
        description="Whether to include a 5-day weather forecast"
    )

In [10]:
@tool(args_schema=WeatherInput)
def get_weather(
    location: str,
    units: str = "celsius",
    include_forecast: bool = False
) -> str:
    """
    Get current weather information and optional 5-day forecast.
    """

    # Mock weather data
    temp = 22 if units == "celsius" else 72
    unit_symbol = "°C" if units == "celsius" else "°F"

    result = (
        f"Current weather in {location}: "
        f"{temp}{unit_symbol}"
    )

    if include_forecast:
        result += (
            "\n\n5-Day Forecast:"
            "\nDay 1: Sunny"
            "\nDay 2: Partly Cloudy"
            "\nDay 3: Rainy"
            "\nDay 4: Windy"
            "\nDay 5: Clear Sky"
        )

    return result

In [11]:
get_weather.invoke(
    {
        "location": "Boston"
    }
)

'Current weather in Boston: 22°C'